In [143]:
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score


In [158]:
data_path = "data/framingham.csv"
df = pd.read_csv(data_path)
print(len(df))
print(df.isna().sum().sum())
df = pd.read_csv(data_path).dropna()
print(df.isna().sum().sum())
print(len(df))

4238
645
0
3656


In [145]:
df_vars = df.drop(columns=['TenYearCHD'])
df_heart_attack_10y = df["TenYearCHD"]

In [156]:
df_vars.head(10)

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose
0,1,48,4.0,1,20.0,0.0,0,1,0,259.0,135.0,90.0,20.72,102.0,81.0
1,1,40,4.0,0,0.0,0.0,0,1,0,240.0,150.0,98.0,40.38,70.0,74.0
2,1,58,1.0,0,0.0,0.0,0,0,0,233.0,125.5,84.0,26.05,67.0,76.0
3,1,49,3.0,0,0.0,0.0,0,1,0,267.0,160.5,109.0,28.33,70.0,75.0
4,0,50,1.0,1,30.0,0.0,0,0,0,203.0,128.5,82.0,18.99,55.0,84.0
5,0,53,4.0,0,0.0,0.0,0,1,0,310.0,146.0,91.0,29.30,75.0,72.0
6,1,45,1.0,1,20.0,0.0,0,0,0,264.0,118.5,81.0,26.35,75.0,90.0
7,1,49,3.0,1,15.0,0.0,0,0,0,208.0,118.0,73.0,24.16,75.0,75.0
8,0,40,1.0,1,20.0,0.0,0,0,0,272.0,123.0,75.0,23.08,80.0,63.0
9,1,53,2.0,0,0.0,0.0,0,1,0,198.0,142.5,82.0,23.84,57.0,78.0


In [147]:
count = (df_heart_attack_10y == 0).sum()
print(count)
count = (df_heart_attack_10y == 1).sum()
print(count)

#há 3099 casos sem ataque cardiaco e 557 ataques cardiacos, um dataset completamente desbalanceado

3099
557


In [148]:
df_classe_0 = df[df['TenYearCHD'] == 0]
df_classe_1 = df[df['TenYearCHD'] == 1]

df_classe_0_under = df_classe_0.sample(len(df_classe_1), random_state=42)

df_balanced = pd.concat([df_classe_0_under, df_classe_1], axis=0)

df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

df_vars = df_balanced.drop(columns=['TenYearCHD'])
df_vars_with_const= sm.add_constant(df_vars)
df_heart_attack_10y = df_balanced['TenYearCHD']


In [149]:
X_train, X_test, y_train, y_test = train_test_split(
    df_vars_with_const, df_heart_attack_10y, test_size=0.25, random_state=42)

In [150]:
model = sm.Logit(y_train, X_train).fit()
params = model.params


Optimization terminated successfully.
         Current function value: 0.593924
         Iterations 6


In [151]:
df_prediction = pd.DataFrame()
df_prediction["probability_of_heart_attack_10y"] = model.predict(X_test)
df_prediction["heart_attack_10y"] = (df_prediction["probability_of_heart_attack_10y"] > 0.5).astype(int)
df_prediction.insert(2, "total_logit_z", model.predict(X_test, linear=True))
df_parameters_z = X_test.multiply(model.params)
df_parameters_z = df_parameters_z.add_prefix('logit_comp_')

df_prediction = pd.concat([df_prediction, df_parameters_z], axis=1)




/home/davi/logistic-regression-heart-disease/venv/lib/python3.10/site-packages/statsmodels/discrete/discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)


In [152]:
print(df_prediction.head())

      probability_of_heart_attack_10y  heart_attack_10y  total_logit_z  \
879                          0.212416                 0      -1.310424   
101                          0.267827                 0      -1.005676   
1111                         0.522732                 1       0.090991   
726                          0.369771                 0      -0.533200   
291                          0.292560                 0      -0.882982   

      logit_comp_const  logit_comp_male  logit_comp_age  logit_comp_education  \
879          -6.674445              0.0        3.160910             -0.212326   
101          -6.674445              0.0        4.163149             -0.212326   
1111         -6.674445              0.0        4.548626             -0.070775   
726          -6.674445              0.0        4.240245             -0.212326   
291          -6.674445              0.0        2.775433             -0.141551   

      logit_comp_currentSmoker  logit_comp_cigsPerDay  logit_comp_BP

In [153]:
accuracy = accuracy_score(y_test, df_prediction["heart_attack_10y"])
print("accuracy", accuracy)
precision = precision_score(y_test, df_prediction["heart_attack_10y"])
print("precision", precision)
recall = recall_score(y_test, df_prediction["heart_attack_10y"])
print("recall", recall)

accuracy 0.6200716845878136
precision 0.5714285714285714
recall 0.7131782945736435


In [154]:
#balanceamento 

